In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
_os.environ['AWS_REGION'] = 'us-east-1'
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions handed explicit creds (e.g. assume_role into other
            # accounts); only bare/SDK-created sessions get the refreshable creds.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# Nova Data Mixing

Data mixing blends your custom training data with Nova's curated synthetic datasets
(code, math, chat, planning, instruction-following, reasoning, etc.) to prevent
catastrophic forgetting while specializing the model on your domain.

> **Important:** Data mixing is only supported with **serverless** compute type.
> It is not available for serverful training jobs (SMTJ) or HyperPod clusters.

## What you will learn

1. Configure `DataMixingConfig` with customer and Nova data percentages
2. Create an `SFTTrainer` with data mixing enabled
3. Set hyperparameters and submit a training job
4. Monitor job status

## 1. Setup

In [2]:
import json
import boto3

In [3]:
# === Fill in your AWS resources ===
REGION = "us-east-1" # e.g. "us-east-1"
ROLE_ARN = "arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole"
S3_BUCKET = "notebook-test-engine-ds-674622101542-use1" # e.g. "sagemaker-us-east-1-674622101542"

S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/sft-data-mixing/output"
TRAINING_DATASET = "s3://notebook-test-engine-ds-674622101542-use1/datasets/sample-converse-messages.jsonl"

## 2. Configure Data Mixing

Data mixing controls the blend between your custom training data and Nova's internal
curated datasets. `customer_data_percent` sets how much of the training data comes from
your dataset. The remaining portion is distributed among Nova categories according to
`nova_data_percentages`.

Available Nova categories include: `code`, `math`, `chat`, `planning`,
`instruction-following`, `reasoning`, `stem`, `rag`, `factuality`, etc.

In [4]:
from sagemaker.train.data_mixing_config import DataMixingConfig

# 70% of training data from your dataset, 30% from Nova curated data
# Within Nova data: 30% code, 70% math
data_mixing_config = DataMixingConfig(
    customer_data_percent=70.0,
    nova_data_percentages={
        "code": 30.0,
        "math": 70.0,
    },
)

## 3. Create SFTTrainer with Data Mixing

Pass the `DataMixingConfig` to `SFTTrainer`. Since data mixing only works with
serverless compute, no `compute` parameter is needed.

In [5]:
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.train import SFTTrainer
from sagemaker.train.common import TrainingType

sft_trainer = SFTTrainer(model_package_group=ModelPackageGroup.create(model_package_group_name="nte-model-package-group"), accept_eula=True, 
    model="amazon.nova-2-lite-v1",
    training_type=TrainingType.LORA,
    training_dataset=TRAINING_DATASET,
    s3_output_path=S3_OUTPUT_PATH,
    role=ROLE_ARN,
    data_mixing_config=data_mixing_config,
    base_job_name="sft-datamix",
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/29/26 08:27:13] INFO     Creating model_package_group resource.                              ]8;id=7149312;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7149313;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23134\23134]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=7149320;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=7149321;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=7149327;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=7149328;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

[07/29/26 08:27:15] INFO     SageMaker session not provided. Using default Session.                  ]8;id=7149335;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=7149336;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

## 4. Set Hyperparameters and Submit Training Job

In [6]:
# Set hyperparameters
sft_trainer.hyperparameters.max_steps = 50
sft_trainer.hyperparameters.learning_rate = 5e-6
sft_trainer.hyperparameters.global_batch_size = 32

# Submit (non-blocking)
training_job = sft_trainer.train(wait=False)
print(f"Training job submitted: {training_job}")

[07/29/26 08:27:16] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7149343;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7149344;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/29/26 08:27:17] INFO     SageMaker session not provided. Using default Session.                  ]8;id=7149349;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=7149350;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=7149357;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=7149358;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=7149364;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=7149365;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Training Job Name: sft-datamix-20260729082717                       ]8;id=7149372;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/sft_trainer.py\sft_trainer.py]8;;\:]8;id=7149373;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/sft_trainer.py#324\324]8;;\

...

[07/29/26 08:27:21] INFO     Resolved datamix recipe for model                             ]8;id=7149416;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/data_mixing_utils.py\data_mixing_utils.py]8;;\:]8;id=7149417;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/data_mixing_utils.py#457\457]8;;\
                             'nova-textgeneration-lite-v2' (multimodal=False): 23                                  
                             categories found.                                                                     

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


                    INFO     Creating training_job resource.                                     ]8;id=7149423;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7149424;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31123\31123]8;;\

Training job submitted: training_job_name='sft-datamix-20260729082717' training_job_arn='arn:aws:sagemaker:us-east-1:674622101542:training-job/sft-datamix-20260729082717' tuning_job_arn=Unassigned() labeling_job_arn=Unassigned() auto_ml_job_arn=Unassigned() model_artifacts=Unassigned() training_job_status='InProgress' secondary_status='Starting' failure_reason=Unassigned() hyper_parameters={'adam_beta1': '0.9', 'adam_beta2': '0.95', 'alpha': '64', 'customer_data_percent': '70', 'data_s3_path': 's3://notebook-test-engine-ds-674622101542-use1/datasets/sample-converse-messages.jsonl', 'fine_tuned_model': '1.0', 'global_batch_size': '32', 'learning_rate': '5e-06', 'learning_rate_ratio': '64.0', 'limit_val_batches': '2', 'lora_alpha': '64', 'lora_plus_lr_ratio': '64.0', 'lr': '5e-06', 'max_context_length': '8192', 'max_length': '32768', 'max_steps': '50', 'min_lr': '1e-06', 'mlflow_experiment_name': 'my-lora-sft-experiment', 'mlflow_run_name': 'my-lora-sft-run', 'model_name_or_path': 'nova-

## 5. Monitor Training Job

In [7]:
from sagemaker.core.resources import TrainingJob

job = TrainingJob.get(training_job_name=training_job.training_job_name)
print(f"Status: {job.training_job_status}")
print(f"Secondary Status: {job.secondary_status}")

Status: InProgress
Secondary Status: Starting


## 6. Alternative: Different Data Mix Configurations

Here are some common configuration patterns depending on your use case.

In [8]:
# High specialization: mostly your data
high_specialization = DataMixingConfig(
    customer_data_percent=90.0,
    nova_data_percentages={
        "reasoning": 100.0,
    },
)

# Balanced: equal split with multiple Nova categories
balanced_mix = DataMixingConfig(
    customer_data_percent=50.0,
    nova_data_percentages={
        "code": 40.0,
        "reasoning": 30.0,
        "math": 30.0,
    },
)

# Light specialization: preserve broad capabilities
light_specialization = DataMixingConfig(
    customer_data_percent=30.0,
    nova_data_percentages={
        "code": 25.0,
        "math": 25.0,
        "chat": 25.0,
        "reasoning": 25.0,
    },
)

## Tips

- **High customer_data_percent (80–90%)** — Use when your task is well-defined and you have enough data.
- **Balanced (50–70%)** — Good default for most use cases.
- **Low customer_data_percent (20–40%)** — Preserve base model capabilities with light specialization.
- **Nova category selection** — Choose categories that complement your task (e.g., `code` + `reasoning` for a coding assistant).